In [13]:
prompt_template = '''
You are a Jira ticket quality evaluator.

Evaluate the following ticket based on these criteria:
1. Problem clarity
2. Reproduction steps
3. Environment details
4. Impact
5. Evidence (logs/screenshots)

For each criterion:
- Mark: PRESENT / PARTIAL / MISSING
- Give a short reason

Then:
- Assign a score out of 8
- Classify as:
  COMPLETE (7–8)
  PARTIAL (4–6)
  INCOMPLETE (0–3)

Finally:
- List missing information
- Suggest what user should add

Return response in JSON format:

{{
  "score": number,
  "classification": "..."
}}

Ticket:
{ticket_data}
'''

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",  # or llama3-8b-8192
    temperature=0
)

In [4]:
jira_ticket_text = '''
    Title: Login failing for some users

Description:
Some users are unable to login to the system.

Steps to Reproduce:
1. Go to login page
2. Enter credentials
3. Click login

Expected Behavior:
User should be logged in.

Actual Behavior:
Login fails with error message "Something went wrong"

Environment:
- Environment: Production

'''

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(prompt_template)

print(prompt)



input_variables=['ticket_data'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['ticket_data'], input_types={}, partial_variables={}, template='\nYou are a Jira ticket quality evaluator.\n\nEvaluate the following ticket based on these criteria:\n1. Problem clarity\n2. Reproduction steps\n3. Environment details\n4. Impact\n5. Evidence (logs/screenshots)\n\nFor each criterion:\n- Mark: PRESENT / PARTIAL / MISSING\n- Give a short reason\n\nThen:\n- Assign a score out of 8\n- Classify as:\n  COMPLETE (7–8)\n  PARTIAL (4–6)\n  INCOMPLETE (0–3)\n\nFinally:\n- List missing information\n- Suggest what user should add\n\nReturn response in JSON format:\n\n{{\n  "score": number,\n  "classification": "..."\n}}\n\nTicket:\n{ticket_data}\n'), additional_kwargs={})]


In [17]:
chain = prompt | llm

response = chain.invoke({
    "ticket_data": jira_ticket_text
})

response

AIMessage(content='{\n  "score": 2,\n  "classification": "INCOMPLETE",\n  "missing_information": [\n    "Specific users affected (e.g., usernames, roles, or IDs)",\n    "Detailed error information (full error message, stack trace, error codes)",\n    "Browser, OS, and version details",\n    "Severity and business impact (how many users, any downtime, SLA breach)",\n    "Logs, screenshots, or any other evidence showing the failure"\n  ],\n  "suggestions": [\n    "Add a list of example accounts that cannot log in, or describe the pattern (e.g., all users from a certain domain).",\n    "Include the exact error message text, any error codes, and a screenshot of the failure dialog.",\n    "Specify the browsers and operating systems used when reproducing the issue, along with their versions.",\n    "Explain the impact: is it affecting all users, a specific group, or causing critical downtime?",\n    "Attach relevant logs from the server or client, or provide a link to a log file."\n  ]\n}', 

In [ ]:
response.